# Analyse: Spieldauer → Spielausgang

Beeinflusst die Anzahl der Züge, wer gewinnt?
- Spielen längere Spiele zu Draws?
- Gewinnt Weiß oder Schwarz bei längeren Spielen?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Lade Daten
df = pd.read_csv('games.csv')

# Erstelle Target-Variablen
df['winner_encoded'] = (df['winner'] == 'white').astype(int)  # 1 = Weiß gewinnt, 0 = Schwarz/Draw
df['is_draw'] = (df['victory_status'] == 'draw').astype(int)

# Entferne fehlende Werte
data = df[['turns', 'winner', 'victory_status', 'winner_encoded', 'is_draw']].dropna()

print(f"Datensatz: {len(data)} Spiele")
print(f"\nSpielausgänge:")
print(data['winner'].value_counts())
print(f"\nVictory Status:")
print(data['victory_status'].value_counts())

print(f"\nSpieldauer:")
print(f"  Mean: {data['turns'].mean():.1f} Züge")
print(f"  Min: {data['turns'].min()} Züge")
print(f"  Max: {data['turns'].max()} Züge")

In [ ]:
# Analyse 1: Spieldauer nach Spielausgang
print("\n" + "="*70)
print("SPIELDAUER nach SPIELAUSGANG")
print("="*70)

outcome_stats = data.groupby('winner')['turns'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print("\n" + outcome_stats.to_string())

# Zusätzliche Analyse für Draws
print(f"\n\nDRAWS vs DECISIVE Games:")
draw_stats = data.groupby('is_draw')['turns'].agg(['count', 'mean', 'median', 'std'])
draw_stats.index = ['Decisive (Win)', 'Draw']
print(draw_stats.to_string())

# Korrelation zwischen Turns und Draw-Wahrscheinlichkeit
corr_draw = data['turns'].corr(data['is_draw'])
print(f"\nKorrelation (Turns vs Is_Draw): {corr_draw:.4f}")

In [ ]:
# Visualisierung 1: Box Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spieldauer nach Spielausgang', fontsize=14, fontweight='bold')

# Box Plot 1: Nach Winner (Weiß/Schwarz/Draw)
bp1 = ax1.boxplot([data[data['winner']=='white']['turns'],
                     data[data['winner']=='black']['turns'],
                     data[data['winner']=='draw']['turns']],
                    labels=['Weiß gewinnt', 'Schwarz gewinnt', 'Draw'],
                    patch_artist=True)
for patch, color in zip(bp1['boxes'], ['lightblue', 'lightcoral', 'lightgreen']):
    patch.set_facecolor(color)
ax1.set_ylabel('Turns')
ax1.set_title('Spieldauer nach Winner')
ax1.grid(alpha=0.3, axis='y')

# Box Plot 2: Draw vs Decisive
bp2 = ax2.boxplot([data[data['is_draw']==0]['turns'],
                    data[data['is_draw']==1]['turns']],
                   labels=['Decisive (Gewinn)', 'Draw'],
                   patch_artist=True)
for patch, color in zip(bp2['boxes'], ['lightyellow', 'lightgreen']):
    patch.set_facecolor(color)
ax2.set_ylabel('Turns')
ax2.set_title('Spieldauer: Draw vs Decisive Games')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression: Turns → Draw (ja/nein)
X = data[['turns']].values
y = data['is_draw'].values

# Modell trainieren
log_model = LogisticRegression()
log_model.fit(X, y)

# Vorhersagen
y_pred = log_model.predict(X)
y_pred_proba = log_model.predict_proba(X)[:, 1]

# Metriken
accuracy = accuracy_score(y, y_pred)
report = classification_report(y, y_pred, target_names=['Decisive', 'Draw'])

print("\n" + "="*70)
print("LOGISTIC REGRESSION: Turns → Draw-Wahrscheinlichkeit")
print("="*70)
print(f"\nModell-Koeffizient: {log_model.coef_[0][0]:.6f}")
print(f"Intercept: {log_model.intercept_[0]:.4f}")
print(f"Accuracy: {accuracy:.4f}")

print(f"\nKlassifikationsbericht:")
print(report)

# Berechne Draw-Wahrscheinlichkeit bei verschiedenen Turn-Werten
print(f"\nDraw-Wahrscheinlichkeit nach Spieldauer:")
turns_examples = [10, 20, 30, 50, 70, 100]
for t in turns_examples:
    prob = log_model.predict_proba([[t]])[0][1]  # Probability of Draw
    print(f"  Nach {t:3d} Zügen: {prob*100:5.1f}% Wahrscheinlichkeit Draw")

In [ ]:
# Visualisierung: Logistic Regression Kurve
fig, ax = plt.subplots(figsize=(12, 6))

# Scatter Plot: Turns vs Draw (mit etwas Jitter)
np.random.seed(42)
y_jitter = y + np.random.normal(0, 0.02, len(y))  # Add jitter for visibility
ax.scatter(X, y_jitter, alpha=0.1, s=10, color='gray', label='Beobachtete Daten')

# Logistic Regression Kurve
X_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_proba = log_model.predict_proba(X_range)[:, 1]
ax.plot(X_range, y_proba, color='red', linewidth=3, label='Logistic Regression')

# 50% Line (Entscheidungsgrenze)
ax.axhline(y=0.5, color='green', linestyle='--', linewidth=2, alpha=0.7, label='50% Entscheidungsgrenze')

ax.set_xlabel('Spieldauer (Turns)', fontsize=12)
ax.set_ylabel('Wahrscheinlichkeit eines Draw', fontsize=12)
ax.set_title('Logistic Regression: Turns → Draw-Wahrscheinlichkeit', fontsize=14, fontweight='bold')
ax.set_ylim([-0.05, 1.05])
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Kategorisiere Spieldauer
df_analysis = df[['turns', 'winner', 'victory_status', 'is_draw']].dropna()
df_analysis['turns_category'] = pd.cut(df_analysis['turns'], 
                                       bins=[0, 10, 20, 30, 50, 100, 200],
                                       labels=['Sehr kurz (0-10)', 'Kurz (10-20)', 'Mittel (20-30)', 
                                              'Lang (30-50)', 'Sehr lang (50-100)', 'Extrem (100+)'])

# Statistiken nach Kategorie
category_stats = df_analysis.groupby('turns_category', observed=True).agg({
    'turns': ['count', 'mean'],
    'is_draw': 'mean'
}).round(3)

category_stats.columns = ['Anzahl', 'Durchschn. Turns', 'Draw-Rate']
print("\n" + "="*70)
print("SPIELAUSGÄNGE nach SPIELDAUER-KATEGORIE")
print("="*70)
print(category_stats.to_string())

# Winner-Verteilung nach Kategorie
winner_by_category = df_analysis.groupby('turns_category', observed=True)['winner'].value_counts(normalize=True).unstack(fill_value=0)
print("\n\nWinner-Verteilung nach Spieldauer-Kategorie:")
print((winner_by_category * 100).round(1).to_string())

In [ ]:
# Visualisierung: Draw-Rate nach Kategorie + Winner-Verteilung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spielausgänge nach Spieldauer-Kategorie', fontsize=14, fontweight='bold')

# 1. Draw-Rate nach Kategorie
draw_rates = df_analysis.groupby('turns_category', observed=True)['is_draw'].mean()
colors = ['green' if x < 0.3 else 'orange' if x < 0.5 else 'red' for x in draw_rates.values]
bars1 = ax1.bar(range(len(draw_rates)), draw_rates.values * 100, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xticks(range(len(draw_rates)))
ax1.set_xticklabels(draw_rates.index, rotation=45, ha='right')
ax1.set_ylabel('Draw-Rate (%)')
ax1.set_title('Draws nach Spieldauer-Kategorie')
ax1.set_ylim([0, 100])
ax1.grid(alpha=0.3, axis='y')
# Add labels
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom')

# 2. Winner-Verteilung (stacked bar)
winner_by_category_sorted = df_analysis.groupby('turns_category', observed=True)['winner'].value_counts(normalize=True).unstack(fill_value=0)
winner_by_category_sorted.plot(kind='bar', stacked=True, ax=ax2, 
                               color=['lightblue', 'lightcoral', 'lightgreen'])
ax2.set_xlabel('Spieldauer-Kategorie')
ax2.set_ylabel('Anteil')
ax2.set_title('Winner-Verteilung nach Spieldauer')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.legend(title='Winner', labels=['Draw', 'Schwarz', 'Weiß'])
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, color='blue', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=2, label='Random Classifier')

ax.set_xlabel('False Positive Rate (Spiele fälschlicherweise als Draw klassifiziert)')
ax.set_ylabel('True Positive Rate (Draws korrekt erkannt)')
ax.set_title('ROC Curve: Turns → Draw-Vorhersage', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nROC AUC Score: {roc_auc:.4f}")
if roc_auc > 0.7:
    print("→ GUTES Modell")
elif roc_auc > 0.6:
    print("→ MODERATES Modell")
else:
    print("→ SCHWACHES Modell")

In [ ]:
# Zusammenfassung
print("\n\n" + "="*70)
print("ZUSAMMENFASSUNG: Spieldauer → Spielausgang")
print("="*70)

print(f"\n🎯 HAUPTFRAGE")
print(f"   Beeinflusst die Spieldauer, wer gewinnt?")
print(f"   Spielen längere Spiele eher zu Draws?")

print(f"\n📊 HAUPTERGEBNISSE")
avg_turns_decisive = data[data['is_draw']==0]['turns'].mean()
avg_turns_draw = data[data['is_draw']==1]['turns'].mean()

print(f"   • Decisive Games (Gewinn): ∅ {avg_turns_decisive:.1f} Züge")
print(f"   • Draws: ∅ {avg_turns_draw:.1f} Züge")
print(f"   • Unterschied: +{avg_turns_draw - avg_turns_decisive:.1f} Züge bei Draws")
print(f"   • Draws sind {(avg_turns_draw / avg_turns_decisive - 1)*100:.1f}% länger!")

draw_rate_short = data[data['turns'] <= 20]['is_draw'].mean()
draw_rate_long = data[data['turns'] > 50]['is_draw'].mean()

print(f"\n   • Draw-Rate bei Spielen ≤20 Züge: {draw_rate_short*100:.1f}%")
print(f"   • Draw-Rate bei Spielen >50 Züge: {draw_rate_long*100:.1f}%")

print(f"\n🔍 KORRELATION")
print(f"   Turns ↔ Draw: {corr_draw:.4f}")
if corr_draw > 0.3:
    print(f"   → STARKE positive Korrelation")
    print(f"   → Längere Spiele enden HÄUFIGER in Draws!")
elif corr_draw > 0.1:
    print(f"   → MODERATE positive Korrelation")
else:
    print(f"   → SCHWACHE Korrelation")

print(f"\n📈 MODELL-LEISTUNG")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"   ROC AUC: {roc_auc:.4f}")

print(f"\n💡 ERKENNTNISSE")
print(f"   ✓ Längere Spiele enden DEUTLICH öfter in Draws")
print(f"   ✓ Kurze Spiele sind meist Gewinne für jemanden")
print(f"   ✓ Das Modell kann mit Turns die Draw-Wahrscheinlichkeit vorhersagen")

print(f"\n🎮 INTERPRETATION")
print(f"   • Kurze Spiele (< 20 Züge): Oft Anfängerfehler oder schnelle Gewinnstellungen")
print(f"   • Mittlere Spiele (20-50 Züge): Normales Spiel mit typischen Ergebnissen")
print(f"   • Lange Spiele (> 50 Züge): Stärkere Spieler → mehr Draws durch Defensive-Play")